In [ ]:
'''
python version 3.10.12
'''


In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import argparse
import json
import os
import sys
import warnings
from importlib import import_module
from pathlib import Path
from shutil import copy
from typing import Dict, List, Union

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
# from torch.utils.tensorboard import SummaryWriter
# from torchcontrib.optim import SWA
from torch.optim.swa_utils import SWALR, AveragedModel, update_bn

from data_utils import (
    Dataset_ASVspoof2019_train,
    Dataset_ASVspoof2019_devNeval,
    genSpoof_list,
)
from evaluation import calculate_tDCF_EER
from my_utils import create_optimizer, seed_worker, set_seed, str_to_bool

warnings.filterwarnings("ignore", category=FutureWarning)

import torch
import pexpect
import matplotlib.pyplot as plt
import time
import json
from pathlib import Path

In [2]:
def get_model(model_config: Dict, device: torch.device):
    """Define DNN model architecture"""
    module = import_module("models.{}".format(model_config["architecture"]))
    _model = getattr(module, "Model")
    model = _model(model_config).to(device)
    nb_params = sum([param.view(-1).size()[0] for param in model.parameters()])
    print("no. model params:{}".format(nb_params))

    return model


def get_loader(
    database_path: str, eval_pro_path:str,seed: int, config: dict
) -> List[torch.utils.data.DataLoader]:
    eval_trial_path = (
        eval_pro_path
    )
    file_eval = genSpoof_list(dir_meta=eval_trial_path, is_train=False, is_eval=True)
    '''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
    '''
    eval_set = Dataset_ASVspoof2019_devNeval(
        list_IDs=file_eval, base_dir=Path('path to VoiceWukong dataset')
    )
    eval_loader = DataLoader(
        eval_set,
        batch_size=config["batch_size"],
        shuffle=False,
        drop_last=False,
        pin_memory=True,
    )

    return eval_loader


import time
def produce_evaluation_file(
    data_loader: DataLoader,
    model,
    device: torch.device,
    save_path: str,
    trial_path: str,

) -> None:
    global timestamps, memory_usage,gpu_utilization

    model.eval()
    with open(trial_path, "r") as f_trl:
        trial_lines = f_trl.readlines()
    fname_list = []
    score_list = []
    
    for batch_x, utt_id in data_loader:
        batch_x = batch_x.to(device)
        with torch.no_grad():
            what, batch_out = model(batch_x)
            batch_score = (batch_out[:, 1]).data.cpu().numpy().ravel()
            

        fname_list.extend(utt_id)
        score_list.extend(batch_score.tolist())
   
    
    assert len(trial_lines) == len(fname_list) == len(score_list)

    with open(save_path, "w") as fh:
        for fn, sco, trl in zip(fname_list, score_list, trial_lines):
            _, utt_id, _, src, key = trl.strip().split(" ")
            assert fn == utt_id
            fh.write("{} {} {} {}\n".format(utt_id, src, key, sco))
    print("Scores saved to {}".format(save_path))


def train_epoch(
    trn_loader: DataLoader,
    model,
    optim: Union[torch.optim.SGD, torch.optim.Adam],
    device: torch.device,
    scheduler: torch.optim.lr_scheduler,
    config: argparse.Namespace,
):
    running_loss = 0
    num_total = 0.0
    ii = 0
    model.train()


    weight = torch.FloatTensor([0.1, 0.9]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    for batch_x, batch_y in trn_loader:
        batch_size = batch_x.size(0)
        num_total += batch_size
        ii += 1
        batch_x = batch_x.to(device)
        batch_y = batch_y.view(-1).type(torch.int64).to(device)
        _, batch_out = model(batch_x, Freq_aug=str_to_bool(config["freq_aug"]))
        batch_loss = criterion(batch_out, batch_y)
        running_loss += batch_loss.item() * batch_size
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        if config["optim_config"]["scheduler"] in ["cosine", "keras_decay"]:
            scheduler.step()
        elif scheduler is None:
            pass
        else:
            raise ValueError("scheduler error, got:{}".format(scheduler))

    running_loss /= num_total
    return running_loss


In [3]:



def main(eval_score_path,eval_pro_path) -> None:
    """
    please change this path
    """

    with open('path to/VoiceWukong/aasist/config/AASIST.conf', "r") as f_json:
        config = json.loads(f_json.read())

    model_config = config["model_config"]
    optim_config = config["optim_config"]

    optim_config["epochs"] = config["num_epochs"]
    track = config["track"]

    assert track in ["LA", "PA", "DF"], "Invalid track given"
    if "eval_all_best" not in config:
        config["eval_all_best"] = "True"
    if "freq_aug" not in config:
        config["freq_aug"] = "False"

    # make experiment reproducible
    set_seed(1234, config)

    # define database related paths
    # output_dir = Path(output_dir)
    # prefix_2019 = "ASVspoof2019.{}".format(track)
    """
    please change this path
    """
    '''
    - path to VoiceWukong dataset
    - - Alldataset
    - - Alldataset32K
    - - ...
    '''
    database_path = Path('path to VoiceWukong dataset')
    # dev_trial_path = (
    #     database_path
    #     / "ASVspoof2019_{}_cm_protocols/{}.cm.dev.trl.txt".format(track, prefix_2019)
    # )

    eval_trial_path = (
        Path(eval_pro_path)
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device: {}".format(device))
    if device == "cpu":
        raise ValueError("GPU not detected!")
    
    model = get_model(model_config, device)

    # define dataloaders
    eval_loader = get_loader(database_path,eval_pro_path, 1234, config)

    
    model.load_state_dict(torch.load(config["model_path"], map_location=device))

    print("Model loaded : {}".format(config["model_path"]))
    print("Start evaluation...")

    produce_evaluation_file(
        eval_loader, model, device, eval_score_path, eval_trial_path
    )

    
    print("DONE.")


In [ ]:

main('path to save eval_score.txt','path to VoiceWukong/aasist/eval_list.txt')
main('path to save zh_eval_score.txt','path to VoiceWukong/aasist/zh_eval_list.txt')